# 🏭 MA-GRPO: Sinh Bộ Dữ Liệu Train GRPO Chuẩn `Condition D` (Macro-Only)
### Batch vLLM Pipeline cho Google Colab Pro (NVIDIA A100 40GB / 80GB)

**Mục tiêu:** Dùng Teacher Planner **`Qwen/Qwen2.5-32B-Instruct-AWQ`** tạo bộ `grpo_train_condition_d.jsonl` (**80 truyện × 8 chương = 640 chapter prompts**) sạch, khớp 100% format prompt của **Condition D** (`VLLMWritingPolicy.construct_prompt`), có `prev_chapter_tail` viết bằng văn xuôi thật (bỏ câu mock), và tách biệt hoàn toàn với tập test `tell-me-a-story-test.jsonl` để huấn luyện **Writing Agent `Qwen/Qwen2.5-7B-Instruct` (QLoRA GRPO)**.

**Thời gian dự kiến trên A100 (40GB/80GB):** ~30–45 phút với `32B-AWQ` (Batch vLLM gom `8` truyện/lượt + tự động dùng `float16` cho AWQ).

---

## Bước 1: Kiểm tra GPU A100 & Môi trường

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name} ({vram:.1f} GB VRAM) | BF16: {torch.cuda.is_bf16_supported()}")


## Bước 2: Mount Google Drive & Clone Mã Nguồn `MA-GRPO`
*Lưu `GH_TOKEN` trong Colab Secrets (biểu tượng chìa khóa bên trái) để chạy không cần nhập tay.*

In [ ]:
import os
import sys
import subprocess
import getpass
from google.colab import drive

drive.mount('/content/drive')
OUTPUT_DIR = "/content/drive/MyDrive/MA-GRPO-Dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Thư mục lưu dataset trên Drive: {OUTPUT_DIR}")

REPO_URL = "github.com/AIVIETNAM-AIO-TruongDogki/MA_GRPO.git"
REPO_DIR = "/content/MA-GRPO"

try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None

if not os.path.exists(REPO_DIR):
    if token:
        subprocess.run(["git", "clone", f"https://{token}@{REPO_URL}", REPO_DIR], check=True)
    else:
        res = subprocess.run(["git", "clone", f"https://{REPO_URL}", REPO_DIR])
        if res.returncode != 0:
            token = getpass.getpass("Nhập GitHub Personal Access Token (repo private): ")
            subprocess.run(["git", "clone", f"https://{token}@{REPO_URL}", REPO_DIR], check=True)
else:
    print(f"Cập nhật mã nguồn mới nhất tại {REPO_DIR}...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

assert os.path.exists("scripts/generate_grpo_dataset.py"), "Không tìm thấy scripts/generate_grpo_dataset.py!"
print("Working dir:", os.getcwd())
!git log -n 1 --oneline


## Bước 3: Cài đặt `vllm` & Thư viện cần thiết

In [ ]:
!pip install -q vllm pyyaml pydantic
print("✅ Cài đặt hoàn tất!")


## Bước 4: Sinh `640` Chapter Prompts Chuẩn `Condition D` (Batch vLLM + Auto-Resume)
- **Nguồn đề bài:** `data/mops_premises.jsonl` + `CANONICAL_20_PREMISES` (tự động loại bỏ mọi đề bài trùng với `data/tell-me-a-story-test.jsonl`).
- **Quy mô mặc định:** `80` truyện × `8` chương = **`640` chapter prompts**, ngân sách độ dài rải đều `{8000, 9000, 10000}` từ/truyện (`~1,000–1,250` từ/chương).
- **Lưu dần theo từng batch** lên Drive (`--resume`): nếu lỡ đứt kết nối giữa chừng, chạy lại cell này sẽ nối tiếp ngay từ batch kế tiếp.

In [ ]:
import subprocess
import sys

MODEL_ID = "Qwen/Qwen2.5-32B-Instruct-AWQ"
NUM_PREMISES = 80          # 80 truyện x 8 chương = 640 chapter prompts
NUM_CHAPTERS = 8
BATCH_SIZE = 8             # 8 truyện chạy song song an toàn KV-cache cho 32B-AWQ trên A100 40GB
OUTPUT_FILENAME = "grpo_train_condition_d.jsonl"

cmd = [
    sys.executable, "-u", "scripts/generate_grpo_dataset.py",
    "--backend", "vllm",
    "--model", MODEL_ID,
    "--premises-file", "data/mops_premises.jsonl",
    "--exclude-test-file", "data/tell-me-a-story-test.jsonl",
    "--num-premises", str(NUM_PREMISES),
    "--num-chapters", str(NUM_CHAPTERS),
    "--target-words-choices", "8000", "9000", "10000",
    "--batch-size", str(BATCH_SIZE),
    "--gpu-memory-utilization", "0.88",
    "--max-model-len", "12288",
    "--prose-max-tokens", "900",
    "--output-dir", OUTPUT_DIR,
    "--output-filename", OUTPUT_FILENAME,
    "--resume",
]

print("Running command:\n", " ".join(cmd), "\n", flush=True)
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"generate_grpo_dataset.py exited with code {ret}")
print("\n✅ Sinh dataset hoàn tất!")


## Bước 5: Kiểm tra Chất lượng Dataset & Tương thích 100% với `scripts/train.py`

In [ ]:
import json
from collections import Counter
from pathlib import Path
from scripts.train import (
    extract_context_from_prompt,
    extract_tail_from_prompt,
    extract_target_words_from_text,
)

dataset_path = Path(OUTPUT_DIR) / OUTPUT_FILENAME
report_path = Path(OUTPUT_DIR) / "dataset_quality_report.json"

print("=== DATASET QUALITY REPORT ===")
if report_path.exists():
    print(report_path.read_text(encoding="utf-8"))

rows = [json.loads(line) for line in dataset_path.read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"\nTổng số chapter prompts đọc được: {len(rows)}")

# Kiểm tra khớp parser của scripts/train.py trên toàn bộ mẫu
target_ok = sum(1 for r in rows if extract_target_words_from_text(r["prompt"]) == r["target_words"])
ctx_ok = sum(1 for r in rows if bool(extract_context_from_prompt(r["prompt"])))
ch2_rows = [r for r in rows if r["chapter_index"] > 1]
tail_ok = sum(1 for r in ch2_rows if bool(extract_tail_from_prompt(r["prompt"])))
tail_lens = [len(r["prev_chapter_tail"].split()) for r in ch2_rows]

print(f"Parser target_words match : {target_ok}/{len(rows)}")
print(f"Parser context_summary ok : {ctx_ok}/{len(rows)}")
print(f"Parser prev_tail (Ch2+) ok: {tail_ok}/{len(ch2_rows)} | avg tail words: {sum(tail_lens)/max(1,len(tail_lens)):.1f}")
print("Phân bố Genre             :", dict(Counter(r.get("genre") for r in rows)))
print("Phân bố Chapter Index     :", dict(sorted(Counter(r.get("chapter_index") for r in rows).items())))

print("\n--- PREVIEW MẪU CHƯƠNG 2 ---")
sample_ch2 = next((r for r in rows if r["chapter_index"] == 2), rows[0])
print(sample_ch2["prompt"][:1200])


## Bước 6: Tự động Ngắt Server Colab Sau Khi Xong (Tiết kiệm Compute Units)
*Dữ liệu đã được lưu an toàn trong Google Drive (`/content/drive/MyDrive/MA-GRPO-Dataset/grpo_train_condition_d.jsonl`). Đổi `AUTO_DISCONNECT = False` nếu bạn muốn giữ runtime lại để kiểm tra thêm.*

In [ ]:
import time

AUTO_DISCONNECT = True

if AUTO_DISCONNECT:
    print("✅ Hoàn tất sinh dataset! Chờ 15 giây để Google Drive đồng bộ tệp rồi tự ngắt runtime...")
    time.sleep(15)
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print(f"Không thể gọi runtime.unassign(): {e}")
else:
    print("ℹ️ AUTO_DISCONNECT=False — Runtime vẫn đang mở. Nhớ tắt thủ công khi xong để tránh tốn Compute Units.")
